In [1]:

!pip install -q langchain langchain-groq


In [2]:

import os
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("Groq API key connected successfully!")


Groq API key connected successfully!


In [3]:

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.3
)

print("Groq model initialized successfully!")


Groq model initialized successfully!


In [4]:

response = llm.invoke("Hello! Introduce yourself.")

print(response.content)


Hello! I’m ChatGPT, an AI language model created by OpenAI. I’m here to help answer your questions, brainstorm ideas, explain concepts, draft text, solve problems, and chat about a wide range of topics. Whether you need assistance with writing, learning something new, troubleshooting a technical issue, or just a friendly conversation, feel free to let me know how I can help!


In [5]:

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

from langchain_core.output_parsers import StrOutputParser


In [6]:

SYSTEM = """ROLE: You are the friendly assistant for Sweet Crumbs Home Bakery.

FACTS: Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
Orders need 24 hours' notice. All cakes have eggless versions.

RULES: Never promise a delivery time or place. If it isn't in FACTS,
say "I'll check with the owner." Work out totals carefully.

FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""


In [7]:

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

print("Prompt and chain created successfully!")


Prompt and chain created successfully!


In [8]:

history = []

def ask(question: str) -> str:

    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer


In [9]:

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))

print(ask("What's my name?"))


Hi Ravi! 😊 Yes, we offer eggless versions of all our cakes—including plum, chocolate, and cupcakes. Let us know what you’d like, and we’ll get it ready for you!
Your name is Ravi. 😊


In [10]:

import csv

def run_tests(path="/content/test.csv"):

    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    passed = 0

    for row in rows:

        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        expected = row["must_include"].lower()
        ok = expected in answer.lower()

        if ok:
            passed += 1

        print(
            "PASS" if ok else "FAIL",
            "|",
            row["question"],
            "->",
            answer[:100]
        )

    print(
        f"\n{passed}/{len(rows)} passed "
        f"({passed / len(rows):.0%})"
    )

run_tests()


PASS | How much is a plum cake? -> A plum cake costs ₹500. Let me know if you’d like to place an order or need any other details!
FAIL | Do you make eggless cakes? -> Yes! All our cakes—including plum, chocolate, and cupcakes—are available in egg‑less versions. Let u
FAIL | Can I get a cake today evening? -> I’ll check with the owner and get back to you soon!
PASS | Do you deliver to Kakinada? -> I'll check with the owner and get back to you soon!
PASS | What is the price of 6 cupcakes? -> Sure! Six cupcakes would cost ₹480 (₹80 each). Let me know if you’d like to place an order!
PASS | What is the price of a chocolate cake? -> The chocolate cake costs Rs 650. Let me know if you’d like to place an order!
FAIL | How much are 2 plum cakes? -> Two plum cakes will be Rs 1,000. 😊  
(We have eggless options, and orders need a 24‑hour notice.)
PASS | What is the price of 3 cupcakes? -> Three cupcakes cost ₹240 (₹80 × 3). Let me know if you'd like to add anything else!
PASS | Can you promise d

In [11]:

from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

print("Agent created successfully!")


Agent created successfully!


In [12]:

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Do you make eggless cakes?"
        }
    ]
})

print(result["messages"][-1].content)


Yes! All our cakes – plum, chocolate and cupcakes – are available in egg‑less versions. Let us know which one you’d like!


In [13]:

IMPROVED_SYSTEM = SYSTEM + """

ADDITIONAL INSTRUCTIONS:
- Always mention the exact price when asked about products.
- Calculate quantity-based totals correctly.
- Mention 24 hours' notice for same-day orders.
- For delivery questions, reply exactly:
  "I'll check with the owner."
- Confirm that all cakes have eggless versions.
- Never invent delivery information.
"""


In [14]:

improved_prompt = ChatPromptTemplate.from_messages([
    ("system", IMPROVED_SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

improved_chain = improved_prompt | llm | StrOutputParser()

print("Improved chain created successfully!")


Improved chain created successfully!


In [16]:

def run_improved_tests(path="/content/test.csv"):

    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    passed = 0

    for row in rows:

        answer = improved_chain.invoke({
            "history": [],
            "question": row["question"]
        })

        expected = row["must_include"].lower()
        ok = expected in answer.lower()

        if ok:
            passed += 1

        print(
            "PASS" if ok else "FAIL",
            "|",
            row["question"],
            "->",
            answer[:150]
        )

    print(f"\nIMPROVED RESULT: {passed}/{len(rows)} passed")

run_improved_tests()


PASS | How much is a plum cake? -> Our plum cake costs Rs 500, and we also offer an eggless version for the same price. 😊
FAIL | Do you make eggless cakes? -> Yes! All our cakes—including plum, chocolate, and cupcakes—are available in egg‑less versions. Let us know what you’d like, and we’ll get it ready for
FAIL | Can I get a cake today evening? -> I'll check with the owner.
PASS | Do you deliver to Kakinada? -> I'll check with the owner.
PASS | What is the price of 6 cupcakes? -> 6 cupcakes cost Rs 480. All our cakes, including cupcakes, are available in eggless version. Please note we need 24 hours’ notice for orders.
PASS | What is the price of a chocolate cake? -> The chocolate cake is Rs 650. 😊 All our cakes, including the chocolate one, are available in eggless versions.
FAIL | How much are 2 plum cakes? -> Two plum cakes cost Rs 1,000. All our cakes, including plum cake, are available in eggless version. Please place the order at least 24 hours in advanc
PASS | What is the pric

In [17]:

print("Original test result : 6/10")
print("Improved test result : See result above")


Original test result : 6/10
Improved test result : See result above
